# Track Deleted Works (oxjob #784)

Maintains the durable ledger of works that have disappeared from
`openalex.works.openalex_works`. Runs right after Guardrails, so a run that
loses works pathologically is stopped before it can ledger the loss.

Two tables:

- **`openalex.works.deleted_works`** — the ledger. One row per (work, deletion):
  `work_id` (BIGINT), `deleted_date` (first date the work was detected gone),
  `deleted_at` (timestamp), `es_deleted_at` (stamped by
  `notebooks/elastic/delete_works` once the ES doc is removed; NULL = ES delete
  still pending). Consumed by the ES delete task and by the daily snapshot's
  `deleted_ids.csv` export.
- **`openalex.works.openalex_works_id_snapshot`** — the id set of
  `openalex_works` as of the last successful run of this notebook. Tonight's
  deletions = snapshot ∖ current. Rewritten at the end of every run, only after
  the ledger insert succeeded. First run seeds it and ledgers nothing.

**Resurrections**: by policy (tombstone purge below) a ledgered id should never
reappear — a returning record mints a fresh id. The cleanup here is the safety
net for paths that bypass the registries (e.g. location curations): any work
alive in `openalex_works` is dropped from the ledger, because a live work must
never be listed as deleted. A revived work's ES doc restores itself: the vanish
deleted its `openalex_works_hash` row (NOT MATCHED BY SOURCE), so reappearing
re-inserts the hash row with a fresh `updated_date`, which the nightly 2-day
incremental ES sync picks up.

**Guard**: aborts (no ledger insert, snapshot untouched) if `openalex_works` is
empty or tonight's deletions exceed `guard_fraction` (default 0.5%) of the live
count. Guardrails already fails the run upstream at >2M works lost, so tripping
this guard means something slipped past it — investigate before overriding.
Sanctioned mass deletions (#765 drain waves) run with the
`deleted_works_guard_override` job parameter set to `true`.

**A work id has one of three statuses (oxjob #2006)**, read from the view `openalex.works.work_status`:

| status | means | where it comes from |
|---|---|---|
| `live` | the work is served | its id is in `openalex_works` |
| `merged` | its records sit on another live work; the id answers with that work (301) | `merged_work_ids_map` (one row per loser, chains followed to a live winner) |
| `deleted` | its records are gone; the id answers 404 | `deleted_works`, and no live winner in the map |

`deleted_works` ledgers every id that left `openalex_works`, merged or not; a `merged_work_ids` row is what makes it
merged. Paths that only move records (the location matcher's `move`, ad-hoc moves, `RepointWorkIds`) write none, so
the sort below writes it for a work they emptied. A loser that is live again, a chain ending at a dead work and a loop
are left out of the map (rules: `notebooks/elastic/sync_merged_ids`).


In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("deleted_works_guard_override", "false")
dbutils.widgets.text("guard_fraction", "0.005")

# job parameter OR a pre-cleared row in openalex.works.e2e_overrides (scripts/preclear_e2e.py)
GUARD_OVERRIDE = spark.sql(
    f"SELECT openalex.works.e2e_override_active('deleted_works_guard_override', '{dbutils.widgets.get('deleted_works_guard_override').replace(chr(39), chr(39) * 2)}')"
).collect()[0][0]
GUARD_FRACTION = float(dbutils.widgets.get("guard_fraction"))
# a pre-cleared row may cap the override at the planned number of deletions (max_changed); above it the guard fires anyway.
# Read through the one reader, openalex.works.e2e_cap (notebooks/end2end/e2e_gate.sql, oxjob #2005): NULL = no row,
# BIGINT max = an uncapped row, else the smallest open cap.
_cap = spark.sql(
    "SELECT openalex.works.e2e_cap('deleted_works_guard_override', 'works_deleted').cap AS c"
).collect()[0]["c"] if GUARD_OVERRIDE else None
GUARD_CAP = None if _cap is None or _cap == 2**63 - 1 else _cap

CATALOG = "openalex"
WORKS = f"{CATALOG}.works.openalex_works"
LEDGER = f"{CATALOG}.works.deleted_works"
ID_SNAPSHOT = f"{CATALOG}.works.openalex_works_id_snapshot"
MERGED = f"{CATALOG}.works.merged_work_ids"            # loser -> winner ledger (every merge path writes it)
LOCATIONS = f"{CATALOG}.works.locations_mapped"         # record -> work, rebuilt every night (7 days of Delta history)
SORT_AUDIT = f"{CATALOG}.works.vanished_work_sort"      # one row per vanished id with no redirect: moved or gone (#2006)
MERGED_MAP = f"{CATALOG}.works.merged_work_ids_map"    # loser -> live winner, chains followed (notebooks/elastic/sync_merged_ids)

print(f"deleted_works_guard_override: {GUARD_OVERRIDE}" + (f" (capped at {GUARD_CAP:,} deletions)" if GUARD_CAP is not None else ""))
print(f"guard_fraction: {GUARD_FRACTION}")
print(f"works: {WORKS}")


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LEDGER} (
    work_id BIGINT NOT NULL,
    deleted_date DATE NOT NULL,
    deleted_at TIMESTAMP NOT NULL,
    es_deleted_at TIMESTAMP
)
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SORT_AUDIT} (
    run_ts TIMESTAMP, work_id BIGINT, verdict STRING, target_work_id BIGINT,
    n_records BIGINT, n_unplaced BIGINT, n_targets BIGINT, locations_version BIGINT)
CLUSTER BY (work_id)
""")


In [0]:
resurrected = spark.sql(
    f"DELETE FROM {LEDGER} WHERE work_id IN (SELECT id FROM {WORKS})"
).collect()[0].num_affected_rows
print(f"Resurrected works removed from ledger: {resurrected:,}")


In [0]:
current_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {WORKS}").collect()[0].cnt
print(f"Current {WORKS} count: {current_count:,}")

FIRST_RUN = not spark.catalog.tableExists(ID_SNAPSHOT)
if FIRST_RUN:
    if current_count == 0:
        raise Exception(f"ABORT: {WORKS} is empty; refusing to seed {ID_SNAPSHOT} from it.")
    spark.sql(f"CREATE TABLE {ID_SNAPSHOT} AS SELECT id FROM {WORKS}")
    print(f"First run: seeded {ID_SNAPSHOT} with {current_count:,} ids; no deletions ledgered.")
else:
    gone_df = spark.sql(f"""
        SELECT s.id AS work_id
        FROM {ID_SNAPSHOT} s
        LEFT ANTI JOIN {WORKS} w ON s.id = w.id
    """)
    gone_count = gone_df.count()
    print(f"Works gone since last run: {gone_count:,}")

    if current_count == 0:
        raise Exception(
            f"ABORT: {WORKS} has 0 rows; every id would ledger as deleted. "
            "Upstream build failed in a way Guardrails missed. Ledger and snapshot unchanged."
        )
    if gone_count > GUARD_FRACTION * current_count and not GUARD_OVERRIDE:
        raise Exception(
            f"ABORT: {gone_count:,} works gone (> {GUARD_FRACTION:.2%} of {current_count:,} live). "
            "Ledger and snapshot unchanged, so tonight's diff is preserved for inspection. "
            "If this is a sanctioned mass deletion, re-run with deleted_works_guard_override=true."
        )
    if GUARD_CAP is not None and gone_count > GUARD_CAP:
        raise Exception(
            f"ABORT: {gone_count:,} works gone, more than the {GUARD_CAP:,} the pre-clear allowed for. "
            "Ledger and snapshot unchanged. Size the extra deletions before re-running with a larger cap."
        )

    gone_df.createOrReplaceTempView("gone_work_ids")


## Moved or gone: a vanished work whose records all sit on one live work is merged into it (oxjob #2006)

For each id that left `openalex_works` tonight with no `merged_work_ids` row: take its records from last night's
`locations_mapped` (the version current when this notebook last wrote the id snapshot) and look them up in tonight's.
- **moved**: every record that still exists sits on one live work → a `merged_work_ids` row (source `auto_emptied`),
  written before the ledger row, so the id redirects instead of answering 404.
- **gone**: no record of it is left; **split**: its records sit on two or more live works; **unplaced**: a record
  exists but on no live work; **no_records**: it had none in last night's table. These stay deleted.

Every verdict is kept in `vanished_work_sort` (replay and evidence: oxjob #2006). If last night's version is no longer
readable (the End 2 End skipped more than 7 days), the sort is skipped with a warning and every id is ledgered as
before. Revert: `DELETE FROM merged_work_ids WHERE source = 'auto_emptied'`.


In [0]:
if not FIRST_RUN:
    snap_ts = (spark.sql(f"DESCRIBE HISTORY {ID_SNAPSHOT}")
               .where(F.col("operation").startswith("CREATE")).agg(F.max("timestamp").cast("string")).first()[0])
    prev_version = (spark.sql(f"DESCRIBE HISTORY {LOCATIONS}")
                    .where(F.col("timestamp") <= F.lit(snap_ts).cast("timestamp")).agg(F.max("version")).first()[0])
    print(f"Last id snapshot {snap_ts}; last night's {LOCATIONS} = version {prev_version}")
    run_ts = spark.sql("SELECT CAST(current_timestamp() AS STRING) AS t").first().t  # a string: no time-zone round trip
    sorted_ok = False
    try:  # only the time-travel read may fail softly; the redirect write below must raise
        if prev_version is None:
            raise Exception(f"no {LOCATIONS} version at or before {snap_ts}")
        n = spark.sql(f"""
            INSERT INTO {SORT_AUDIT}
            WITH cand AS (
              SELECT g.work_id FROM gone_work_ids g LEFT ANTI JOIN {MERGED} m ON m.loser_work_id = g.work_id
            ),
            old AS (
              SELECT DISTINCT o.work_id, o.provenance, o.native_id_namespace, o.native_id
              FROM {LOCATIONS} VERSION AS OF {prev_version} o JOIN cand c ON c.work_id = o.work_id
            ),
            nw AS (
              SELECT DISTINCT n.provenance, n.native_id_namespace, n.native_id, n.work_id
              FROM {LOCATIONS} n
              LEFT SEMI JOIN old ON n.provenance = old.provenance AND n.native_id_namespace = old.native_id_namespace
                                AND n.native_id = old.native_id
            ),
            rec AS (
              SELECT old.work_id, nw.work_id AS new_work, w.id IS NOT NULL AS new_live
              FROM old
              LEFT JOIN nw ON nw.provenance = old.provenance AND nw.native_id_namespace = old.native_id_namespace
                          AND nw.native_id = old.native_id
              LEFT JOIN {WORKS} w ON w.id = nw.work_id
            ),
            per AS (
              SELECT work_id, COUNT(*) AS n_records,
                     COUNT_IF(new_work IS NOT NULL AND NOT new_live) AS n_unplaced,
                     COUNT(DISTINCT CASE WHEN new_live THEN new_work END) AS n_targets,
                     MIN(CASE WHEN new_live THEN new_work END) AS target
              FROM rec GROUP BY work_id
            ),
            v AS (
              SELECT c.work_id, p.n_records, p.n_unplaced, p.n_targets, p.target,
                     CASE WHEN p.work_id IS NULL THEN 'no_records'
                          WHEN p.n_targets > 1 THEN 'split'
                          WHEN p.n_unplaced > 0 THEN 'unplaced'
                          WHEN p.n_targets = 1 THEN 'moved'
                          ELSE 'gone' END AS verdict
              FROM cand c LEFT JOIN per p ON p.work_id = c.work_id
            )
            SELECT TIMESTAMP '{run_ts}', work_id, verdict, IF(verdict = 'moved', target, NULL),
                   n_records, n_unplaced, n_targets, {prev_version}
            FROM v
        """).collect()[0].num_inserted_rows
        sorted_ok = True
    except Exception as e:
        print(f"WARNING: moved-or-gone sort skipped ({str(e)[:300]}); every vanished id is ledgered as deleted tonight.")
    if sorted_ok:
        spark.sql(f"""
            SELECT verdict, COUNT(*) AS works FROM {SORT_AUDIT} WHERE run_ts = TIMESTAMP '{run_ts}'
            GROUP BY verdict ORDER BY verdict
        """).show()
        # the redirect goes in before the ledger row (next cell), so no reader sees the id deleted without it
        moved = spark.sql(f"""
            MERGE INTO {MERGED} t
            USING (SELECT work_id AS loser_work_id, target_work_id AS winner_work_id FROM {SORT_AUDIT}
                   WHERE run_ts = TIMESTAMP '{run_ts}' AND verdict = 'moved') s
            ON t.loser_work_id = s.loser_work_id AND t.winner_work_id = s.winner_work_id
            WHEN NOT MATCHED THEN INSERT (loser_work_id, winner_work_id, merged_at, source)
            VALUES (s.loser_work_id, s.winner_work_id, TIMESTAMP '{run_ts}', 'auto_emptied')
        """).collect()[0].num_inserted_rows
        print(f"Sorted {n:,} vanished works with no redirect; {moved:,} moved -> {MERGED} (auto_emptied).")


In [0]:
if not FIRST_RUN:
    inserted = spark.sql(f"""
        INSERT INTO {LEDGER}
        SELECT g.work_id, current_date(), current_timestamp(), NULL
        FROM gone_work_ids g
        LEFT ANTI JOIN {LEDGER} l ON g.work_id = l.work_id
    """).collect()[0].num_inserted_rows
    print(f"Ledgered {inserted:,} deletions (dated {spark.sql('SELECT current_date() AS d').collect()[0].d}).")

    spark.sql(f"CREATE OR REPLACE TABLE {ID_SNAPSHOT} AS SELECT id FROM {WORKS}")
    print(f"Snapshot rewritten with {current_count:,} ids.")

pending = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {LEDGER} WHERE es_deleted_at IS NULL"
).collect()[0].cnt
total = spark.sql(f"SELECT COUNT(*) AS cnt FROM {LEDGER}").collect()[0].cnt
print(f"Ledger: {total:,} total deletions, {pending:,} pending ES delete.")


## Tombstone purge — a ledgered id stays dead (policy: Casey, 2026-08-17)

Once a work is ledgered as deleted, its identity rows are purged from BOTH
registries — `work_id_map` (resolution keys) and `location_work_ids` (anchor
verdicts). If the underlying record later re-enters harvest, nothing resolves,
so MapWorkIds mints a fresh id: the dead id never returns. This is the standing
exception to the registries' insert-only discipline (noted in MapWorkIds).

Keyed on the full ledger every night, so it is idempotent and self-backfilling
(the first run purges the pre-policy backlog). Undo path: both tables carry
60-day Delta retention, so a wrong purge is recoverable via `VERSION AS OF`
within that window (mistakes surface within days via the guard and morning
verification).

The resurrection cleanup above stays as a safety net: paths that write explicit
work_ids without consulting the registries (e.g. location curations) can still
revive a ledgered id, and the invariant "a live work never appears in the
ledger or deleted_ids.csv" must hold regardless.


In [0]:
for src, key in (
    (f"{CATALOG}.works.work_id_map", "id"),
    (f"{CATALOG}.works.location_work_ids", "work_id"),
):
    purged = spark.sql(
        f"DELETE FROM {src} WHERE {key} IN (SELECT work_id FROM {LEDGER})"
    ).collect()[0].num_affected_rows
    print(f"{src}: purged {purged:,} identity rows for ledgered work ids")


## One work status (oxjob #2006)
The view every reader uses for "is this work id live, merged into X, or deleted?" (states: top of this notebook).


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.works.work_status
COMMENT 'oxjob #2006: one row per published work id: live, merged (merged_into = its live winner) or deleted. Precedence live > merged > deleted. Recreated by notebooks/end2end/TrackDeletedWorks.'
AS
SELECT id AS work_id, 'live' AS status, CAST(NULL AS BIGINT) AS merged_into, CAST(NULL AS DATE) AS status_date
FROM {WORKS}
UNION ALL
SELECT m.id, 'merged', m.merge_into_id, m.merge_date
FROM {MERGED_MAP} m
LEFT ANTI JOIN {WORKS} w ON w.id = m.id
UNION ALL
SELECT d.work_id, 'deleted', CAST(NULL AS BIGINT), MIN(d.deleted_date)
FROM {LEDGER} d
LEFT ANTI JOIN {WORKS} w ON w.id = d.work_id
LEFT ANTI JOIN {MERGED_MAP} m ON m.id = d.work_id
GROUP BY d.work_id
""")
print(f"{CATALOG}.works.work_status recreated")
